## Implement Page Rank Algorithm. (Use python or beautiful soup for implementation).

## 📚 PageRank Algorithm — Formula & Working

---

## ✅ What is PageRank?
PageRank is an algorithm developed by Google to rank web pages in search results based on their importance.

It models the web as a **directed graph**:
- Each **webpage is a node**.
- Each **hyperlink is a directed edge** from one node to another.

---

## ✅ 📊 PageRank Formula

For a page **P**, its PageRank is computed as:

$$
PR(P) = \frac{1 - d}{N} + d \sum_{Q \in In(P)} \frac{PR(Q)}{L(Q)}
$$

Where:
- **PR(P)** = PageRank of page **P**  
- **d** = Damping factor (usually **0.85**)  
- **N** = Total number of pages  
- **In(P)** = Set of pages that link to page **P**  
- **L(Q)** = Number of outbound links from page **Q**

---

## ✅ 📖 Intuition Behind the Formula
- Each page gets a **base rank**:  
  $$
  \frac{1 - d}{N}
  $$
  This models the probability of a random jump.
  
- Plus contributions from other pages linking to it:
  $$
  d \times \sum_{Q \in In(P)} \frac{PR(Q)}{L(Q)}
  $$

- The damping factor **d = 0.85** accounts for the user continuing to click links.

---

## ✅ 🛠️ Step-by-Step Working Example

### Example:
We have **3 web pages: A, B, C**
- A links to B and C  
- B links to C  
- C links to A  

#### Initial State:
$$
PR(A) = PR(B) = PR(C) = \frac{1}{3}
$$

---

### 1️⃣ First Iteration (with d = 0.85)

$$
PR(A) = \frac{1 - 0.85}{3} + 0.85 \times \frac{PR(C)}{1}
$$

$$
PR(B) = \frac{1 - 0.85}{3} + 0.85 \times \frac{PR(A)}{2}
$$

$$
PR(C) = \frac{1 - 0.85}{3} + 0.85 \times \left( \frac{PR(A)}{2} + \frac{PR(B)}{1} \right)
$$

We repeat the above calculation multiple times until the values stabilize (converge).

---

### 2️⃣ Convergence
We continue iterations until the **change is less than a small threshold (epsilon)**.

---

## ✅ 📊 Why PageRank Works Well
- Pages linked from **many important pages get a higher rank**.
- Prevents spam: Pages with no inbound links get low rank.
- Based purely on the **link structure of the web**.

---

## ✅ 📈 Visual Graph Example



In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import numpy as np
import time

MAX_PAGES = 10
MAX_DEPTH = 2

def get_links(url):
    try:
        soup = BeautifulSoup(requests.get(url, timeout=5).content, 'html.parser')
        links = {urljoin(url, a['href']).split('#')[0].rstrip('/')
                 for a in soup.find_all('a', href=True)
                 if urlparse(a['href']).scheme in ('http', 'https') or a['href'].startswith('/')}
        return links
    except:
        return set()

def crawl(seed_url):
    visited, graph, queue = set(), {}, [(seed_url, 0)]
    while queue and len(visited) < MAX_PAGES:
        url, depth = queue.pop(0)
        if url in visited or depth > MAX_DEPTH:
            continue
        print(f"Crawling: {url}")
        visited.add(url)
        links = get_links(url) - {url}
        graph[url] = links
        queue.extend((link, depth + 1) for link in links if link not in visited)
        time.sleep(1)
    return graph

def pagerank(graph, d=0.85, max_iter=100, tol=1e-6):
    nodes = list(graph.keys())
    n = len(nodes)
    ranks = np.ones(n) / n
    idx = {url: i for i, url in enumerate(nodes)}
    M = np.zeros((n, n))

    for i, url in enumerate(nodes):
        out_links = graph[url]
        if not out_links:
            M[:, i] = 1 / n
        else:
            for link in out_links:
                if link in idx:
                    M[idx[link], i] = 1 / len(out_links)

    for _ in range(max_iter):
        new_ranks = (1 - d) / n + d * M @ ranks
        if np.linalg.norm(new_ranks - ranks, 1) < tol:
            break
        ranks = new_ranks

    return dict(zip(nodes, ranks))

if __name__ == '__main__':
    seed = 'https://w3schools.com'
    graph = crawl(seed)

    print("\nGraph:")
    for url, links in graph.items():
        print(f"{url} -> {len(links)} links")

    print("\nPageRank Scores:")
    scores = pagerank(graph)
    for url, score in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"{url}: {score:.4f}")


Crawling: https://w3schools.com
Crawling: https://w3schools.com/about/default.asp
Crawling: https://w3schools.com/html/tryit.asp?filename=tryhtml_default_default
Crawling: https://w3schools.com/vue/index.php
Crawling: https://w3schools.com/js/tryit.asp?filename=tryjs_default
Crawling: https://w3schools.com/bootstrap/bootstrap_quiz.asp
Crawling: https://campus.w3schools.com/products/learn-typescript
Crawling: https://w3schools.com/go/go_exercises.php
Crawling: https://w3schools.com/cpp/default.asp
Crawling: https://w3schools.com/gen_ai/index.php

Graph:
https://w3schools.com -> 277 links
https://w3schools.com/about/default.asp -> 270 links
https://w3schools.com/html/tryit.asp?filename=tryhtml_default_default -> 5 links
https://w3schools.com/vue/index.php -> 270 links
https://w3schools.com/js/tryit.asp?filename=tryjs_default -> 5 links
https://w3schools.com/bootstrap/bootstrap_quiz.asp -> 270 links
https://campus.w3schools.com/products/learn-typescript -> 28 links
https://w3schools.com/g